# SVHN-Detektionsbeispiele (Inline im Notebook)

Nach dem SVHN-Training/Messen werden **10 zufaellige** SVHN-Test-Szenen durch
die volle Pipeline gejagt (Conv+ANFIS-Saliency -> hybrid2-Fenster -> BNN mit
MC-Dropout + Box-Head) und die Bounding-Boxen direkt hier im Notebook
ausgegeben:

- **gruen** = Ground-Truth-Box (mit Ziffern-Label)
- **rot** = Pipeline-Detektion (Klasse + Konfidenz)

Das Konfidenzgatter ist fuer die SVHN-Modelle auf `gate=0.2` gestellt (der
Foto-Klassifikator ist deutlich unsicherer als der MNIST-Szene-Klassifikator).


In [ ]:
import os, sys, glob
import numpy as np
import torch
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from IPython.display import display

def _find_pipeline():
    d = os.getcwd()
    for _ in range(6):
        p = os.path.join(d, 'pipeline')
        if os.path.exists(os.path.join(p, 'evaluate_pipeline.py')):
            return p
        nd = os.path.dirname(d)
        if nd == d:
            break
        d = nd
    return None

PIPE = _find_pipeline()
if PIPE is None:
    raise SystemExit('pipeline/ nicht gefunden (Notebook liegt unter Code/SVHN_Detektionsbeispiele.ipynb erwartet)')
if PIPE not in sys.path:
    sys.path.insert(0, PIPE)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GATE = 0.2
print('Pipeline:', PIPE)
print('Device  :', DEVICE)
print('gate    :', GATE)


## Daten + Modelle laden

Test-Szenen aus `scene_dataset_svhn/` (1000 Bilder, aus sichbaren SVHN-Test-
Ziffern ohne Sprite-Leak gebaut), Modelle aus `pipeline/models/`.


In [ ]:
from data_common import load_scene_split
from stage12 import ConvANFISSaliency
from bnn import BNN
import evaluate_pipeline as E

def _find_svhn_scenes():
    d = os.getcwd()
    for _ in range(6):
        p = os.path.join(d, 'scene_dataset_svhn')
        if os.path.exists(os.path.join(p, 'scene_test.npz')):
            return p
        nd = os.path.dirname(d)
        if nd == d:
            break
        d = nd
    return None

SVHN_DIR = _find_svhn_scenes()
if SVHN_DIR is None:
    raise SystemExit('scene_dataset_svhn nicht gefunden - build_svhn_scenes.py ausfuehren.')
te = load_scene_split('test', base=SVHN_DIR)
print('Test-Szenen:', te['images'].shape, '->', SVHN_DIR)

MODELS = os.path.join(PIPE, 'models')
saliency = ConvANFISSaliency()
saliency.load_state_dict(torch.load(os.path.join(MODELS, 'conv_anfis_saliency_svhn.pt'),
    map_location='cpu', weights_only=False)['model'])
bnn = BNN(n_class=11, box_head=True, c1=40, c2=80, hid=192)
bnn.load_state_dict(torch.load(os.path.join(MODELS, 'bnn_mc_box_svhn.pt'),
    map_location='cpu', weights_only=False)['model'])
print('SVHN-Modelle geladen.')


## 10 zufaellige Szenen: Pipeline laufen lassen, Boxen anzeigen

Jede Ausfuehrung waehlt andere 10 Bilder (seed=None). Saliency wird je Einzel-
bild berechnet, dann hybrid2-Fenster -> BNN-MC (8 Passes) -> Boxen.


In [ ]:
n = 10
figs = E.sample_detection_figures(saliency, bnn, te['images'], te['boxes'], te['labels'],
                                  device=DEVICE, n=n, gate=GATE)
print(f'{len(figs)} zufaellige SVHN-Szenen dargestellt')
for f in figs:
    f.suptitle('SVHN-Detektion (gruen=GT, rot=Pipeline)')
    display(f)
    plt.close(f)


## Einordnung

Die Saliency findet die Ziffern (gruen gedeckt), und die Boxen sitzen im
Mittel ~2.4px genau - aber der Klassifikator verwechselt auf echten Fotos
viele Klassen (digit_acc 0.256), daher sind rote Label haeufig falsch und
es kommen oefter Falsch-Positivs hinzu. Der Engpass liegt nachweislich in
Stufe 4/5 (Klassifikation), nicht in der Co-Lokalisierung.
